In [ ]:
import json
from pathlib import Path
from typing import List, Dict, Set, Tuple
import networkx as nx
import os
from pprint import pprint
import re

os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)

# --- Config ---
PROJECT_ROOT = Path('../..').resolve()  # Adjust as needed to find the project root
GRAPH_PATH = PROJECT_ROOT / "projects/doc_gen/internal/code_graph.gml"
DATABASE_PATH = PROJECT_ROOT / "projects/doc_gen/internal/code_graph.json"
PRIORITY_PATH = PROJECT_ROOT / "projects/doc_gen/internal/forward_pass_schedule.json"
DOX_DIR = PROJECT_ROOT / "projects/doc_gen/internal/generated_dox"
XML_DIR = PROJECT_ROOT / "projects/doc_gen/doxygen_output/xml"

OPENAI_MODEL = "gpt-4.1-nano"
OPENAI_MAX_TOKENS = None

SRC_DIR = PROJECT_ROOT / 'src'
TEMP_DIR = PROJECT_ROOT / 'tmp'

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/qte2333/repos/legacy


In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
from legacy_common.llm_utils import call_openai
from doc_gen.gen_common import get_dependency_docs

In [ ]:
import legacy_common.doxygen_parse as dp

db = dp.load_db(DATABASE_PATH)

In [ ]:
import legacy_common.doxygen_graph as dg

# --- Load graph and priority ---
graph: nx.DiGraph = dg.load_graph(GRAPH_PATH)

2025-06-19 16:14:41.919 | INFO     | doxygen_graph:load_graph:450 - Graph loaded from /Users/qte2333/repos/legacy/projects/doc_gen/internal/code_graph.gml, nodes: 14339, edges: 42761


In [ ]:
import legacy_common.doc_db as doc_db

In [7]:
def extract_function_name(name_str: str) -> str:
    """
    Extract the actual function name from various formats the LLM might return.
    
    Handles:
    - Return types: "String interpret_color_code" -> "interpret_color_code"
    - Parameters: "interpret_color_code(int arg)" -> "interpret_color_code"
    - Qualified names: "ClassName::method" -> "ClassName::method"
    
    Args:
        name_str: The function name string to process
        
    Returns:
        The cleaned function name
    """
    # Remove any parameter list
    if '(' in name_str:
        name_str = name_str.split('(', 1)[0].strip()
    
    # Handle return types by finding the last word
    # But preserve qualified names like Class::method
    parts = name_str.split()
    if len(parts) > 1:
        # Check if we have a qualified name with :: in the last part
        if '::' in parts[-1]:
            return parts[-1].strip()
        
        # Check if the qualified name spans multiple parts
        # e.g., "String Namespace::Class::method"
        for i in range(len(parts)-1, 0, -1):
            if '::' in parts[i]:
                return ' '.join(parts[i:]).strip()
        
        # If we got here, it's likely a return type followed by function name
        return parts[-1].strip()
    
    return name_str.strip()

# Test the function with various examples
test_cases = [
    "interpret_color_code",
    "String interpret_color_code",
    "interpret_color_code(int arg)",
    "String interpret_color_code(int arg)",
    "Class::method",
    "void Class::method",
    "void Class::method(int arg)",
    "Namespace::Class::method",
    "String Namespace::Class::method",
    "String Namespace::Class::method(int arg)"
]

for test in test_cases:
    print(f"{test} -> {extract_function_name(test)}")

interpret_color_code -> interpret_color_code
String interpret_color_code -> interpret_color_code
interpret_color_code(int arg) -> interpret_color_code
String interpret_color_code(int arg) -> interpret_color_code
Class::method -> Class::method
void Class::method -> Class::method
void Class::method(int arg) -> Class::method
Namespace::Class::method -> Namespace::Class::method
String Namespace::Class::method -> Namespace::Class::method
String Namespace::Class::method(int arg) -> Namespace::Class::method


In [16]:
def evaluate_usage_documentation(doc: doc_db.Document):
    """
    Evaluate the usage documentation for a given document.
    
    Args:
        doc: A document with usage information to evaluate
    
    Returns:
        Dictionary of evaluation results
    """
    if not doc or not doc.usages:
        return None
        
    # Extract the function name and brief description
    name = doc.name
    kind = doc.kind
    brief = doc.brief or "No description available"
    
    # Create the prompt for evaluation
    prompt = f"""
You are reviewing how a {kind} is used across a codebase.

The {kind} is:
{name}
"""

    if brief:
        prompt += f"\nIt is summarized as:\n{brief}\n"

    prompt += """
Below are various usage descriptions from other parts of the codebase. Your job is to score each one on:

- Plausibility (0-1): Does it make sense that this usage actually uses this {kind}?
- Specificity (0-1): Does it tell us something unique about how the {kind} is used?

Then suggest whether to:
- keep
- discard
- rewrite (if plausible but poorly phrased or vague)

### Usage examples:
"""

    # Add each usage to the prompt
    for context, usage in doc.usages.items():
        function_name = context.split(', ')[1]
        prompt += f"- {function_name}: \"{usage}\"\n"
    
    prompt += """
### Respond as JSON:
{
  "function_name": {"plausibility": 1.0, "specificity": 0.8, "action": "keep"},
  "another_function": {"plausibility": 0.3, "specificity": 0.2, "action": "discard"},
  "third_function": {"plausibility": 0.9, "specificity": 0.4, "action": "rewrite", "rewrite": "Better description here."}
}
"""
    
    # Call the LLM to evaluate the usages
    try:
        response = call_openai(prompt, OPENAI_MODEL, OPENAI_MAX_TOKENS)
        
        # Parse the JSON response
        import re
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', response)
        if json_match:
            response = json_match.group(1)
            
        # print(prompt)
        evaluation = json.loads(response)
        # pprint(evaluation)
        
        # Process the evaluation results
        updated_usages = {}
        for function_name, result in evaluation.items():
            function_name = extract_function_name(function_name)

            for context, usage in doc.usages.items():
                cid, sig = context.split(', ', 1)
                sig_fname = extract_function_name(sig)

                if sig_fname == function_name:
                    if result["action"] == "keep":
                        updated_usages[context] = usage
                    elif result["action"] == "rewrite" and "rewrite" in result:
                        updated_usages[context] = result["rewrite"]
                    elif result["action"] == "discard":
                        updated_usages[context] = None
                    else:
                        print(f"unknown result for {context}")
                    break
            else:
                print(f"{function_name} not found in usages: {doc.usages.keys()}")
                return None

        # add any that were missed
        for context, usage in doc.usages.items():
            if context not in updated_usages:
                print(f"Missing usage for {context}")
                return
#                updated_usages[context] = usage

        # prune the discarded
        updated_usages = {k:v for k,v in updated_usages.items() if v}

        # don't prune everything though
        if not updated_usages:
            return

        doc.state = doc_db.DocumentState.REFINED_USAGE
        doc.usages = updated_usages 
        doc_db.save_docs(doc.cid)

        return evaluation
    
    except Exception as e:
        print(f"Error evaluating usage documentation: {e}")
        return None

# Test the evaluation function on a few documents
test_docs = []
for cid, docs in doc_db.docs.items():
    for sig, doc in docs.items():
        if doc.usages and doc.state == doc_db.DocumentState.GENERATED_USAGE:
            test_docs.append(doc)

count = 0
for doc in test_docs:
    print(f"\nEvaluating: {doc.cid}, {doc.name}")
    print(f"Brief: {doc.brief}")
    print(f"Usages: {len(doc.usages)}")
    
    evaluation = evaluate_usage_documentation(doc)
    if evaluation:
        print("Evaluation results:")
        for func, result in evaluation.items():
            action = result["action"]
            plausibility = result["plausibility"]
            specificity = result["specificity"]
            print(f"- {func}: P={plausibility:.1f}, S={specificity:.1f}, Action={action}")
            if action == "rewrite":
                print(f"  Rewrite: {result.get('rewrite', 'No rewrite provided')}")
    else:
        print("No evaluation results")

    count += 1
    print(f"{count:>4}/{len(test_docs):>4}", "-" * 40)


Evaluating: remort_8cc, raff_add_to_char
Brief: Adds a remort affect to a character based on a given affect ID.
Usages: 2
Evaluation results:
- load_char_obj: P=1.0, S=0.8, Action=keep
- roll_one_raff: P=1.0, S=0.9, Action=keep
   1/ 982 ----------------------------------------

Evaluating: remort_8cc, roll_one_raff
Brief: Assigns a random remort affect to a character.
Usages: 2
Evaluation results:
- do_raffset: P=1.0, S=0.8, Action=keep
- roll_raffects: P=1.0, S=0.7, Action=keep
   2/ 982 ----------------------------------------

Evaluating: remort_8cc, roll_raffects
Brief: Assigns multiple random remort affects to a character.
Usages: 2
Evaluation results:
- do_remort: P=1.0, S=0.8, Action=keep
- do_raffset: P=1.0, S=0.8, Action=keep
   3/ 982 ----------------------------------------

Evaluating: remort_8cc, HAS_EXTRACLASS
Brief: Checks if a character has access to an extra class skill.
Usages: 6
Evaluation results:
- do_spells: P=1.0, S=0.8, Action=keep
- do_skills: P=1.0, S=0.8, A